# STL activation-range data collection

STL counterpart of `this-and-prev/actiavtion-ranges-data-collection.ipynb`. All
the workhorses now live in `olt.act_ranges.collect`; this notebook is just the
driver: STL model + `stl_transform` + paths. We hook the **pre-relu conv**
(the kernel's own response, before BatchNorm) exactly like the InceptionV1
flow, so nothing BN-specific is needed here.

**Prereq:** download + extract the s0->s4 cluster reports for the STL model
yourself, and point `BASE_REPORT_DIR` at them (they must contain `report.csv`s).

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from olt.act_ranges import collect
from olt.models.stl_inception import stl_inception
from olt.tfms import stl_transform

# --- paths you set --------------------------------------------------------
DEVICE = "mps"
STEM_STRIDE = 2  # must match the checkpoint
# data/stl-bnfree-fancy/checkpoints/step_003120.pt
CKPT_PATH = Path("../../data/stl-bnfree-fancy/checkpoints/step_003120.pt")  # TODO
BASE_REPORT_DIR = Path("stl-mass-train-reports")  # TODO: extracted s0->s4 reports
FLAT_IMAGE_DIR = Path("stl-flat-images")          # TODO: <ikey>.jpeg per firing image
SHARDS_BASE_DIR = Path("../../data/stl/image-shards")         # TODO: for the noise baseline pool
ASSET_DUMP_DIR = Path("act_range_analysis_stl")
ASSET_DUMP_DIR.mkdir(parents=True, exist_ok=True)
FLAT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from olt.df_to_ir import extract_images_to_flat_folder

extract_images_to_flat_folder(SHARDS_BASE_DIR, FLAT_IMAGE_DIR)

In [ ]:
model = stl_inception(ckpt_path=CKPT_PATH, stem_stride=STEM_STRIDE, use_bn=False).to(DEVICE)
model = model.eval()  # BN in inference mode (though we hook the conv, before BN)

## 1. main_df from the downloaded reports

In [ ]:
! ls {BASE_REPORT_DIR}

In [ ]:
main_df = collect.build_main_df(BASE_REPORT_DIR)
main_df.to_csv(ASSET_DUMP_DIR / "main_df.csv")
all_layers = main_df.layer_name.unique()
print(len(main_df), "firings across", list(all_layers))
main_df.head()

## 2. per-cluster neuron activations (conv-output scalar)

In [ ]:
# few STL classes -> take several images per class dir AND several spatial
# locations per image, so the noise pool isn't tiny (~10 samples otherwise).
test_images = collect.get_all_tfmed_images_as_batch(
    SHARDS_BASE_DIR, stl_transform, per_dir=20
)

import pickle
layer_by_channel_by_cid_by_acts = collect.collect_acts_for_all_image(
    main_df, model, all_layers, DEVICE, FLAT_IMAGE_DIR, stl_transform
)
collect.dump_layer_by_chan_by_cid_by_act(layer_by_channel_by_cid_by_acts, ASSET_DUMP_DIR / "layer_by_channel_by_cid_by_acts.pkl")

## 3. noise / baseline activations

In [ ]:
test_images = collect.get_all_tfmed_images_as_batch(SHARDS_BASE_DIR, stl_transform)
layer_by_channel_by_noise_acts = collect.collect_all_noise_activations(
    model, all_layers, test_images, 1, DEVICE
)
import pickle
with open(ASSET_DUMP_DIR / "layer_by_channel_by_noise_acts.pkl", "wb") as f:
    pickle.dump(layer_by_channel_by_noise_acts, f)

In [ ]:
all_layers

In [ ]:
# sanity: scatter one channel's baseline activations
ln = all_layers[0]
data = layer_by_channel_by_noise_acts[ln]["0"]
plt.scatter(range(len(data)), data)
plt.title(f"{ln} chan 0 baseline")
plt.show()

## 4. per-cluster receptive-field input patches

In [ ]:
layer_by_channel_by_cid_by_patches = collect.get_neuron_cluster_patches(
    model,
    main_df,
    FLAT_IMAGE_DIR,
    stl_transform,
    DEVICE,
    checkpoint_path=str(ASSET_DUMP_DIR / "ckpt_layer_by_channel_by_cid_by_patch.pt"),
    checkpoint_every=100,
    max_per_cid=100,
)

In [ ]:
# save per (layer, channel): cid -> stacked patches
import torch
dest_dir = ASSET_DUMP_DIR / "cluster-patches-set"
for layer_name in layer_by_channel_by_cid_by_patches:
    for channel in layer_by_channel_by_cid_by_patches[layer_name]:
        d = dest_dir / layer_name / str(channel) / "cid_by_patches.pt"
        d.parent.mkdir(parents=True, exist_ok=True)
        cid_by_patches = layer_by_channel_by_cid_by_patches[layer_name][channel]
        torch.save({cid: torch.stack(p) for cid, p in cid_by_patches.items()}, d)

# Range analysis — which clusters fire above noise

Loads the two dumps above and, per `(layer, channel)`, scatters each cluster's
activation population against the channel's noise baseline so you can eyeball
which clusters sit on the **higher** side of the activation range.
Re-runnable standalone (reads the pkls), so you don't need to re-run collection.

In [ ]:
import pickle

import pandas as pd

from olt.act_ranges import get_labels_above_noise_range, show_label_points_grid

with open(ASSET_DUMP_DIR / "layer_by_channel_by_cid_by_acts.pkl", "rb") as f:
    ACTS_DICT = pickle.load(f)
with open(ASSET_DUMP_DIR / "layer_by_channel_by_noise_acts.pkl", "rb") as f:
    layer_by_channel_by_noise = pickle.load(f)

main_df = pd.read_csv(ASSET_DUMP_DIR / "main_df.csv")
all_layers = main_df.layer_name.unique()

In [ ]:
# pick a neuron to inspect
LAYER = all_layers[0]
CHANNEL = "0"

label_by_points = ACTS_DICT[LAYER][CHANNEL]  # {cluster_label: [acts]}
noise = layer_by_channel_by_noise[LAYER][CHANNEL]
print(LAYER, "chan", CHANNEL, "-", len(label_by_points), "clusters")

In [ ]:
# all clusters for this neuron, one panel per chunk of labels
show_label_points_grid(label_by_points, 5, 5)

In [ ]:
# just the clusters whose 5th-percentile activation clears the noise range
above = get_labels_above_noise_range(label_by_points, noise, percentile=5)
print("clusters above noise:", list(above.keys()))
if above:
    show_label_points_grid(above, 2, 2)